# Step 1: Build the image dataset

**Goal:** make a labelled set of Amazon-style product photos where some break the listing rules.

| Label | Meaning | 1 = breaks the rule |
|---|---|---|
| `tlgw` | Text, Logo, Graphic, Watermark | something is stamped on top of the photo |
| `nwb` | Non-White Background | background is not pure white (RGB 255,255,255) |

*(Brand infringement and prurience come in phase 2.)*

**Why make our own data?** Nobody publishes thousands of labelled "bad" Amazon photos. So we take clean photos from the free
[Amazon Berkeley Objects](https://amazon-berkeley-objects.s3.amazonaws.com/index.html) dataset and break the rules ourselves.
Because *we* broke the rule, we know the right answer for every image. This is called **synthetic data**.

Runs on Google Colab (free) or your laptop. No AWS account needed.

## 1. Setup
On Colab this downloads the project code. On your laptop, skip the `git clone` line and run the notebook from the `notebooks/` folder.

In [ ]:
import os, sys
if 'google.colab' in sys.modules:
    !git clone -q https://github.com/sujatab1706-tech/my-project.git
    %cd my-project
else:
    os.chdir('..')  # move from notebooks/ to the project root
!pip install -q -r requirements.txt
sys.path.insert(0, 'src')

## 2. Download clean product photos
`src/download_abo.py` downloads product **main images** (the photo that must follow the rules) and keeps only the ones that
already have a white background. About 90% pass that check.

3000 images take about 10 to 15 minutes. Use `--n 500` for a quick test.

In [ ]:
!python src/download_abo.py --n 3000 --out data/raw

In [ ]:
import pandas as pd
raw = pd.read_csv('data/raw/metadata.csv')
print(len(raw), 'clean images')
raw['product_type'].value_counts().head(10)

## 3. Break the rules on purpose
`src/make_violations.py` saves, for every clean photo:
- the **clean** copy (`tlgw=0, nwb=0`)
- one **broken** copy: TLGW only, NWB only, or both

How each violation is made:
- **TLGW:** promo text like "50% OFF" (sometimes on a coloured box), see-through watermarks, or round "NEW" badges.
- **NWB:** a "paint bucket" flood fill from the edges finds the white background, then we swap in a solid colour, a gradient or a blurry texture.

It also splits the data into **train / val / test** (70/15/15) *by product*, so the clean and broken versions of one product
never end up in different splits. Otherwise the model could recognise a product it saw in training and look better than it really is.
That mistake is called **data leakage**.

In [ ]:
!python src/make_violations.py --raw data/raw --out data/processed

## 4. Look at the data
Always look at your data before training. Here are random examples for each kind of violation.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

labels = pd.read_csv('data/processed/labels.csv')

def show(df, title, n=6):
    rows = df.sample(min(n, len(df)), random_state=0)
    fig, axes = plt.subplots(1, len(rows), figsize=(3 * len(rows), 3.3), squeeze=False)
    for ax, r in zip(axes[0], rows.itertuples()):
        ax.imshow(Image.open(f'data/processed/images/{r.file}'))
        ax.set_title(f'tlgw={r.tlgw} nwb={r.nwb}', fontsize=9)
        ax.axis('off')
    fig.suptitle(title)
    plt.show()

show(labels[labels.detail == 'clean'], 'Clean (follows the rules)')
show(labels[(labels.tlgw == 1) & (labels.nwb == 0)], 'TLGW only')
show(labels[(labels.tlgw == 0) & (labels.nwb == 1)], 'NWB only')
show(labels[(labels.tlgw == 1) & (labels.nwb == 1)], 'Both')

### How balanced are the labels?
In classification, if one class is very rare the model can cheat by always guessing the common one.
Each label here is "yes" for roughly a third of the images, which is fine.

In [ ]:
print(labels.groupby('split')[['tlgw', 'nwb']].mean().round(2))
labels['detail'].value_counts()

## 5. Sneak peek: a simple rule for NWB
Before any machine learning, try the simplest thing that could work: **are the pixels along the edge of the photo white?**
(`src/rules.py`). The table below is a **confusion matrix**: rows are the truth, columns are what the rule said.
We'll learn to read these properly in Step 3.

In [ ]:
from rules import is_white_background
test = labels[labels.split == 'test'].copy()
test['rule_says_nwb'] = [int(not is_white_background(Image.open(f'data/processed/images/{f}'))) for f in test.file]
pd.crosstab(test.nwb, test.rule_says_nwb, rownames=['true nwb'], colnames=['rule says nwb'])

The rule is almost perfect here. **Don't celebrate yet:** we made the backgrounds ourselves, so they are very obviously not white.
Real seller photos are harder: soft shadows, off-white studio paper (RGB 245), or a product that touches every edge of the photo.
That's why in Step 2 we also train a CNN, and in Step 3 we test both on harder, real-looking cases.

The rule *does* make a few mistakes: watermarks or banners near the edge make the border look "not white".
Find those images below. Looking at your errors is one of the most useful habits in ML.

In [ ]:
wrong = test[test.nwb != test.rule_says_nwb]
print(len(wrong), 'mistakes')
if len(wrong): show(wrong, 'Where the rule got it wrong', n=len(wrong))

## What's next
**Step 2:** train models. A rule plus OCR as baselines, then one CNN (MobileNetV2 transfer learning) with two outputs, `tlgw` and `nwb`.